In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist

# Load cleaned dataset
df = pd.read_csv("cleaned_transcripts.csv")
df["title"] = df["title"].fillna("")
df["transcript"] = df["transcript"].fillna("")

titles     = df["title"].tolist()
transcripts = df["transcript"].tolist()
video_ids  = df["video_id"].tolist()

print("Total videos:", len(video_ids))
df.head()

c:\Users\Hemanth Sai\OneDrive\Desktop\info\queytube\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total videos: 244


,video_id,title,datetime,transcript
0,1Z-8HxlLAHk,Kids and young people stay curious and be will...,2026-03-30 12:17:27+00:00,like and even at a young age like learning to ...
1,ehkDsePv3W8,Heres a cool and easy way to work with colors ...,2026-03-29 12:43:24+00:00,Working with colors in 3JS One cool way to wor...
2,uWRdzJTpcpI,Do web devs NEED to understand lowlevel progra...,2026-03-28 12:32:40+00:00,A lot of what the curriculum was trying to do ...
3,GC5CLCgnvm0,When things are new and a little scary embraci...,2026-03-27 12:18:22+00:00,what this futures going to look like is not go...
4,tZef2ZzbyuQ,What happens when the model CANT fix it Interv...,2026-03-27 10:00:57+00:00,Welcome back to the Free Code Camp podcast Im ...


In [2]:
# ============================================================
# Build semantic ground truth using a lightweight mapping model
# ============================================================

# Use MiniLM just for mapping (fast, good enough for ground truth)
mapping_model = SentenceTransformer("all-MiniLM-L6-v2")

# Combine title + transcript for richer context
combined_text = [t + " " + tr for t, tr in zip(titles, transcripts)]
doc_embs = mapping_model.encode(combined_text, convert_to_numpy=True, show_progress_bar=True)

print("Document embeddings shape:", doc_embs.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3761.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 8/8 [00:13<00:00,  1.64s/it]


Document embeddings shape: (244, 384)


In [3]:
# Load your existing queries
queries_df = pd.read_csv("queries.csv")
queries = queries_df["query"].tolist()

print("Total queries:", len(queries))
print("Sample:", queries[:5])

Total queries: 75
Sample: ['How to get started with coding for beginners', 'Best way to learn programming in 2025', 'What is the future of web development', 'How to use Three.js for creative coding', 'Beginner guide to JavaScript frameworks']


In [4]:
# Encode all queries
query_embs = mapping_model.encode(queries, convert_to_numpy=True, show_progress_bar=True)

# For each query, find the semantically closest video (cosine similarity)
sim_matrix = cosine_similarity(query_embs, doc_embs)  # shape: (n_queries, n_videos)

mapping_rows = []
for i, query in enumerate(queries):
    best_idx = np.argmax(sim_matrix[i])
    mapping_rows.append({
        "query": query,
        "relevant_video_id": video_ids[best_idx],
        "mapping_score": round(sim_matrix[i][best_idx], 4)
    })

mapping_df = pd.DataFrame(mapping_rows)
mapping_df.to_csv("query_video_map.csv", index=False)

print("Semantic mapping done. Sample:")
print(mapping_df.head(10).to_string())

Batches: 100%|██████████| 3/3 [00:00<00:00, 12.43it/s]


Semantic mapping done. Sample:
                                                  query relevant_video_id  mapping_score
0          How to get started with coding for beginners       lw1vTtFyfgg         0.6305
1                 Best way to learn programming in 2025       Fi8vnYgMiHA         0.6392
2                 What is the future of web development       BiA08jfr4RU         0.5230
3               How to use Three.js for creative coding       v7Butot2fHY         0.6819
4               Beginner guide to JavaScript frameworks       c6IyCwAV6BY         0.5814
5         How to build projects with React step by step       cw02dMpWStI         0.5300
6         What skills do software developers need today       ekXUXu7Eejs         0.5698
7       How to learn full stack development efficiently       cfOI7PBhkWk         0.5050
8                 Tips for mastering Python programming       UsfpzxZNsPo         0.5968
9  What is the best programming language to learn first       xtGThMTO9rI      

In [5]:
# Sanity check: how many unique videos are mapped?
print("Unique videos in ground truth:", mapping_df["relevant_video_id"].nunique())
print("Queries with score > 0.5:", (mapping_df["mapping_score"] > 0.5).sum())
print("Queries with score < 0.3 (low confidence):", (mapping_df["mapping_score"] < 0.3).sum())

# Distribution of top videos
print("\nTop 10 most-mapped videos:")
print(mapping_df["relevant_video_id"].value_counts().head(10))

Unique videos in ground truth: 55
Queries with score > 0.5: 33
Queries with score < 0.3 (low confidence): 1

Top 10 most-mapped videos:
relevant_video_id
ll1mpTJ4qBU    4
mrRfPVm9nAY    3
xHUwGx-ZlTM    3
voPcxCFMbbM    3
suuDVhYXe5Y    3
lw1vTtFyfgg    2
v7Butot2fHY    2
ekXUXu7Eejs    2
UsfpzxZNsPo    2
MdEexYakkHw    2
Name: count, dtype: int64


In [6]:
models = {
    "MiniLM"  : "all-MiniLM-L6-v2",
    "MPNet"   : "all-mpnet-base-v2",
    "MultiQA" : "multi-qa-MiniLM-L6-cos-v1"
}

# Reload ground truth (use the freshly generated semantic mapping)
mapping_df  = pd.read_csv("query_video_map.csv")
queries     = mapping_df["query"].tolist()
ground_truth = dict(zip(mapping_df["query"], mapping_df["relevant_video_id"]))

print("Ground truth loaded. Total queries:", len(queries))

Ground truth loaded. Total queries: 75


In [7]:
results = []

for model_name, model_path in models.items():

    print(f"\n{'='*60}")
    print(f"Loading model: {model_name} ({model_path})")
    print('='*60)

    model = SentenceTransformer(model_path)

    # Step 3: Generate title and transcript embeddings separately
    title_embeddings      = model.encode(titles,      convert_to_numpy=True, show_progress_bar=True)
    transcript_embeddings = model.encode(transcripts, convert_to_numpy=True, show_progress_bar=True)

    print(f"  Title embeddings shape:      {title_embeddings.shape}")
    print(f"  Transcript embeddings shape: {transcript_embeddings.shape}")

    # Combine via average (Step 4)
    doc_embeddings = (title_embeddings + transcript_embeddings) / 2

    # Step 4: Encode queries
    query_embeddings = model.encode(queries, convert_to_numpy=True, show_progress_bar=True)

    # Step 5 (similarity) + Step 6 (distance) — all metrics defined here
    metrics = {
        "cosine"     : lambda q: cosine_similarity(q, doc_embeddings),
        "dot"        : lambda q: np.dot(q, doc_embeddings.T),
        "euclidean"  : lambda q: -cdist(q, doc_embeddings, metric="euclidean"),
        "manhattan"  : lambda q: -cdist(q, doc_embeddings, metric="cityblock"),
        "chebyshev"  : lambda q: -cdist(q, doc_embeddings, metric="chebyshev"),  # WAS MISSING
    }

    for metric_name, metric_func in metrics.items():

        scores = metric_func(query_embeddings)

        top1, top3, top5 = 0, 0, 0
        ranks = []

        # Step 7: Rank videos for each query
        for i, query in enumerate(queries):
            score       = scores[i]
            ranked_idx  = np.argsort(score)[::-1]
            ranked_vids = [video_ids[j] for j in ranked_idx]
            true_video  = ground_truth[query]

            rank = ranked_vids.index(true_video) + 1 if true_video in ranked_vids else len(video_ids) + 1

            ranks.append(rank)
            if rank == 1: top1 += 1
            if rank <= 3: top3 += 1
            if rank <= 5: top5 += 1

        total = len(queries)

        # Step 8: Evaluation metrics
        print(f"\n  [{model_name} | {metric_name}]")
        print(f"  Top-1 Recall : {top1/total:.3f}")
        print(f"  Top-3 Recall : {top3/total:.3f}")
        print(f"  Top-5 Recall : {top5/total:.3f}")
        print(f"  Average Rank : {np.mean(ranks):.2f}")

        results.append({
            "Model"       : model_name,
            "Metric"      : metric_name,
            "Top-1 Recall": round(top1 / total, 4),
            "Top-3 Recall": round(top3 / total, 4),
            "Top-5 Recall": round(top5 / total, 4),
            "Avg Rank"    : round(float(np.mean(ranks)), 2)
        })

print("\nEvaluation complete.")


Loading model: MiniLM (all-MiniLM-L6-v2)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1778.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 8/8 [00:14<00:00,  1.83s/it]


  Title embeddings shape:      (244, 384)
  Transcript embeddings shape: (244, 384)


Batches: 100%|██████████| 3/3 [00:00<00:00, 10.39it/s]



  [MiniLM | cosine]
  Top-1 Recall : 0.520
  Top-3 Recall : 0.840
  Top-5 Recall : 0.907
  Average Rank : 2.64

  [MiniLM | dot]
  Top-1 Recall : 0.560
  Top-3 Recall : 0.773
  Top-5 Recall : 0.893
  Average Rank : 3.00

  [MiniLM | euclidean]
  Top-1 Recall : 0.560
  Top-3 Recall : 0.800
  Top-5 Recall : 0.907
  Average Rank : 2.79

  [MiniLM | manhattan]
  Top-1 Recall : 0.533
  Top-3 Recall : 0.800
  Top-5 Recall : 0.867
  Average Rank : 3.23

  [MiniLM | chebyshev]
  Top-1 Recall : 0.267
  Top-3 Recall : 0.427
  Top-5 Recall : 0.507
  Average Rank : 24.93

Loading model: MPNet (all-mpnet-base-v2)


...


  [MPNet | cosine]
  Top-1 Recall : 0.347
  Top-3 Recall : 0.560
  Top-5 Recall : 0.707
  Average Rank : 6.49

  [MPNet | dot]
  Top-1 Recall : 0.373
  Top-3 Recall : 0.573
  Top-5 Recall : 0.707
  Average Rank : 6.61

  [MPNet | euclidean]
  Top-1 Recall : 0.373
  Top-3 Recall : 0.533
  Top-5 Recall : 0.693
  Average Rank : 7.29

  [MPNet | manhattan]
  Top-1 Recall : 0.320
  Top-3 Recall : 0.507
  Top-5 Recall : 0.653
  Average Rank : 7.93

  [MPNet | chebyshev]
  Top-1 Recall : 0.160
  Top-3 Recall : 0.360
  Top-5 Recall : 0.373
  Average Rank : 36.91

Loading model: MultiQA (multi-qa-MiniLM-L6-cos-v1)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1632.05it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 8/8 [00:30<00:00,  3.86s/it]


  Title embeddings shape:      (244, 384)
  Transcript embeddings shape: (244, 384)


Batches: 100%|██████████| 3/3 [00:00<00:00,  9.22it/s]



  [MultiQA | cosine]
  Top-1 Recall : 0.387
  Top-3 Recall : 0.600
  Top-5 Recall : 0.693
  Average Rank : 8.32

  [MultiQA | dot]
  Top-1 Recall : 0.440
  Top-3 Recall : 0.627
  Top-5 Recall : 0.693
  Average Rank : 8.79

  [MultiQA | euclidean]
  Top-1 Recall : 0.347
  Top-3 Recall : 0.560
  Top-5 Recall : 0.693
  Average Rank : 8.40

  [MultiQA | manhattan]
  Top-1 Recall : 0.360
  Top-3 Recall : 0.520
  Top-5 Recall : 0.640
  Average Rank : 9.44

  [MultiQA | chebyshev]
  Top-1 Recall : 0.213
  Top-3 Recall : 0.373
  Top-5 Recall : 0.507
  Average Rank : 27.92

Evaluation complete.


In [8]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Top-3 Recall", "Avg Rank"], ascending=[False, True])

print("\n===== FULL COMPARISON TABLE =====")
print(results_df.to_string(index=False))

results_df.to_csv("embedding_evaluation_results.csv", index=False)


===== FULL COMPARISON TABLE =====
  Model    Metric  Top-1 Recall  Top-3 Recall  Top-5 Recall  Avg Rank
 MiniLM    cosine        0.5200        0.8400        0.9067      2.64
 MiniLM euclidean        0.5600        0.8000        0.9067      2.79
 MiniLM manhattan        0.5333        0.8000        0.8667      3.23
 MiniLM       dot        0.5600        0.7733        0.8933      3.00
MultiQA       dot        0.4400        0.6267        0.6933      8.79
MultiQA    cosine        0.3867        0.6000        0.6933      8.32
  MPNet       dot        0.3733        0.5733        0.7067      6.61
  MPNet    cosine        0.3467        0.5600        0.7067      6.49
MultiQA euclidean        0.3467        0.5600        0.6933      8.40
  MPNet euclidean        0.3733        0.5333        0.6933      7.29
MultiQA manhattan        0.3600        0.5200        0.6400      9.44
  MPNet manhattan        0.3200        0.5067        0.6533      7.93
 MiniLM chebyshev        0.2667        0.4267        0.

In [9]:
# Primary: highest Top-3 Recall. Tiebreak: lowest Avg Rank.
best_row = results_df.iloc[0]

best_model  = best_row["Model"]
best_metric = best_row["Metric"]

print("===== STEP 10: BEST MODEL & RANKING METHOD =====")
print(f"Best Model   : {best_model}")
print(f"Best Metric  : {best_metric}")
print(f"Top-1 Recall : {best_row['Top-1 Recall']:.3f}")
print(f"Top-3 Recall : {best_row['Top-3 Recall']:.3f}")
print(f"Top-5 Recall : {best_row['Top-5 Recall']:.3f}")
print(f"Average Rank : {best_row['Avg Rank']:.2f}")
print()
print(f"Reason: {best_model} with {best_metric} achieves the highest Top-3 Recall")
print("        with the lowest average rank among tied configurations.")

# Save best config for Module 6 to read
import json
best_config = {
    "model_name" : models[best_model],   # actual HuggingFace model string
    "model_key"  : best_model,
    "metric"     : best_metric
}
with open("best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

print("\nBest config saved to best_config.json (used by Module 6)")
print(json.dumps(best_config, indent=2))

===== STEP 10: BEST MODEL & RANKING METHOD =====
Best Model   : MiniLM
Best Metric  : cosine
Top-1 Recall : 0.520
Top-3 Recall : 0.840
Top-5 Recall : 0.907
Average Rank : 2.64

Reason: MiniLM with cosine achieves the highest Top-3 Recall
        with the lowest average rank among tied configurations.

Best config saved to best_config.json (used by Module 6)
{
  "model_name": "all-MiniLM-L6-v2",
  "model_key": "MiniLM",
  "metric": "cosine"
}
